# Double descent of the QELM on MNIST 3 vs 5

Issue #181. A static, non-linear classification task, MNIST digits 3 against 5, on which the generalization error of the QELM readout shows the double descent around the interpolation threshold `alpha = N_train / N_features = 1`.

The protocol (`experiments/double_descent/double_descent.py`, config `conf/double_descent.yaml`):

- each image is reduced to `N` principal components (PCA fitted on the training images of the two digits), and each component to an angle `theta_i` in `[0, pi]`;
- qubit `i` starts in `R_y(theta_i)|0>`, a product state with no ancilla;
- one fixed quench `U = exp(-i H t)` acts on every image: a single TFIM `H = h sum_i Z_i + sum_{i<j} J_ij X_i X_j`, drawn once per seed, with `h = J` and `t = 10 / J`;
- the features are the one-site `<X_i>, <Y_i>, <Z_i>` and the correlators `<X_i X_j>, <Y_i Y_j>, <Z_i Z_j>` of the final state, plus a bias column: `3 N + 3 N (N - 1) / 2 + 1 = 109` columns at `N = 8`;
- the readout is the unregularized pseudo-inverse `W = Phi^+ y` (`training/regression=pinv`), fitted on `n_train = round(alpha * N_features)` images drawn from a pool, 20 times per `alpha`, and scored on the whole MNIST test split of the two digits.

`N_features` is the **rank** of the feature matrix of the pool, not its number of columns, so that `alpha = 1` is exactly the size where the square system `Phi W = y` becomes solvable. With `alpha = N_train / N_features`, `alpha < 1` is the over-parameterized side (more weights than images, the readout interpolates the training set) and `alpha > 1` the under-parameterized one (least squares).

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_double_descent_notebook.py` does for a small sweep). `N` is the number of qubits, one per principal component. `SEEDS` draws the reservoir (the couplings `J_ij`), the pool and the classical ELM layer. `DATASET=digits` uses the 8x8 digits bundled with scikit-learn instead of MNIST, without download. The sweep goes to this notebook's own folder, `outputs/double_descent/<date_time>/`, and `RUN = False` reads back the latest sweep of this folder.

In [ ]:
N = os.environ.get("N", "8")
SEEDS = os.environ.get("SEEDS", "0 1 2 3 4")
DATASET = os.environ.get("DATASET", "mnist")
N_POOL = os.environ.get("N_POOL", "2000")
N_TEST = os.environ.get("N_TEST", "null")  # null: the whole test split
N_REPEATS = os.environ.get("N_REPEATS", "20")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

One run per seed. The first run downloads MNIST to `data/mnist/` (about 11 MB, gitignored) and checks the SHA-256 of the four files.

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "double_descent"

if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "N": N,
        "SEEDS": SEEDS,
        "DATASET": DATASET,
        "N_POOL": N_POOL,
        "N_TEST": N_TEST,
        "N_REPEATS": N_REPEATS,
        "OUT_DIR": str(sweep_dir),
    }
    run_with_progress(
        ["bash", "experiments/double_descent/double_descent.sh"],
        total=len(SEEDS.split()),
        env=env,
        show=("N", "seed"),
    )
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Accuracies and sanity checks

The readout fitted on the whole pool, scored on the test set, averaged over the seeds, next to the three checks of the issue:

1. **Classical limit.** The classical ELM has as many random ReLU neurons as the QELM has observables, with the same pseudo-inverse readout. The QELM should be comparable to it. `pixels` is the classical reference of the task, a linear readout on all 784 pixels fitted on every training image.
2. **t = 0.** `qelm_t0` measures the same observables on the encoded state, before any dynamics. Its features are checked against their closed form on the product state (`<P_i> = r_i^P`, `<A_i B_j> = r_i^A r_j^B` for the Bloch vectors `r_i`): the largest deviation is printed, 0 up to rounding. So the `t = 0` readout is exactly a linear regression on these fixed functions of the encoded input. Its rank is lower than the QELM's (`<Y_i> = 0` and `<Y_i Y_j> = 0` for a real state), which moves its threshold, and the sweep accounts for it by using the rank.
3. **Peak at alpha = 1.** For each model, the `alpha` where the median test MSE is largest, per seed and for the average over the seeds.

In [ ]:
import importlib.util

# The plotting script of the experiment is not a package: import it by path.
spec = importlib.util.spec_from_file_location(
    "plot_double_descent", "experiments/double_descent/plot_double_descent.py"
)
plot_double_descent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_double_descent)

runs = plot_double_descent.load_runs(sweep_dir)
table = plot_double_descent.summary(runs)
print(f"{table['n_runs']} seeds")
print(f"{'model':>8} {'rank':>10} {'columns':>8} {'accuracy':>17} {'peak alpha':>11}")
for model, row in table["models"].items():
    ranks = sorted(set(row["n_features"]))
    print(
        f"{model:>8} {'/'.join(map(str, ranks)):>10} {row['n_columns']:>8} "
        f"{row['accuracy_mean']:>8.4f} ± {row['accuracy_std']:<6.4f} "
        f"{row['peak_alpha_of_mean']:>11.3g}"
    )
if "accuracy_pixels_mean" in table:
    print(f"pixels: accuracy {table['accuracy_pixels_mean']:.4f}")
print(f"t = 0 features vs closed form: {table['t0_max_deviation']:.1e}")

## Double descent

Test and train errors against `alpha`, the median over the training sets of each seed for the MSE (its tails are heavy near the peak) and the mean for the classification error, then the mean over the seeds with the standard error. The dashed line is `alpha = 1`. Below it the train error is 0: the readout interpolates. The figure and its data are saved in the sweep folder, `double_descent.png` and `double_descent.json`, with `summary.json`.

In [ ]:
import matplotlib.pyplot as plt

table = plot_double_descent.plot_sweep(sweep_dir)
figure_path = sweep_dir / "double_descent.png"
print(f"figure: {figure_path}")

# Draw it again to show it inline (plot_sweep closes its figure).
fig, axes = plot_double_descent.draw(runs)
plt.show()